In [1]:
!pip install -q transformers datasets timm accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00


In [2]:
!pip install kaggle

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/EfficientNetB0/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download patrickaudriaz/tobacco3482jpg
!unzip -q tobacco3482jpg.zip

cp: cannot stat '/content/drive/MyDrive/EfficientNetB0/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/patrickaudriaz/tobacco3482jpg
License(s): other
100%|██████████████████████████████████████| 3.07G/3.07G [02:05<00:00, 26.2MB/s]



In [3]:
import os
import torch
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score

In [4]:
dataset_path = "/kaggle/working/Tobacco3482-jpg"

classes = sorted(os.listdir(dataset_path))

label2id = {cls:i for i, cls in enumerate(classes)}
id2label = {i:cls for cls, i in label2id.items()}

image_paths = []
labels = []

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)

    for img_name in os.listdir(cls_path):

        if img_name.endswith(".jpg"):
            image_paths.append(os.path.join(cls_path, img_name))
            labels.append(label2id[cls])

print("Total Images:", len(image_paths))
print("Classes:", classes)

Total Images: 3482
Classes: ['ADVE', 'Email', 'Form', 'Letter', 'Memo', 'News', 'Note', 'Report', 'Resume', 'Scientific']


In [5]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [6]:
model_name = "microsoft/dit-base-finetuned-rvlcdip"

processor = AutoImageProcessor.from_pretrained(model_name)

preprocessor_config.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

The image processor of type `BeitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [7]:
class TobaccoDataset(Dataset):

    def __init__(self, image_paths, labels, processor):
        self.image_paths = image_paths
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(self.image_paths[idx]).convert("RGB")

        encoding = self.processor(
            images=image,
            return_tensors="pt"
        )

        item = {
            "pixel_values": encoding["pixel_values"].squeeze(),
            "labels": torch.tensor(self.labels[idx])
        }

        return item

In [8]:
train_dataset = TobaccoDataset(
    train_paths,
    train_labels,
    processor
)

val_dataset = TobaccoDataset(
    val_paths,
    val_labels,
    processor
)

In [9]:
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=len(classes),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

BeitForImageClassification LOAD REPORT from: microsoft/dit-base-finetuned-rvlcdip
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([16, 768]) vs model:torch.Size([10, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([16]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [10]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds)
    }

In [11]:
training_args = TrainingArguments(
    output_dir="./dit_tobacco",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=10,

    weight_decay=0.01,

    logging_steps=20,

    load_best_model_at_end=True,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,2.483279,1.901902,0.905308
2,1.178004,0.923512,0.942611
3,0.588858,0.558454,0.956958
4,0.285814,0.428623,0.958393
5,0.301621,0.394894,0.958393
6,0.206909,0.378035,0.958393
7,0.156264,0.359868,0.965567
8,0.141393,0.353009,0.962697
9,0.109323,0.371125,0.962697
10,0.103309,0.363511,0.962697


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1750, training_loss=0.6774486201150077, metrics={'train_runtime': 2962.4986, 'train_samples_per_second': 9.401, 'train_steps_per_second': 0.591, 'total_flos': 2.1585384655865856e+18, 'train_loss': 0.6774486201150077, 'epoch': 10.0})

In [14]:
metrics = trainer.evaluate()
print(metrics)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.35300931334495544, 'eval_accuracy': 0.9626972740315638, 'eval_runtime': 48.903, 'eval_samples_per_second': 14.253, 'eval_steps_per_second': 0.9, 'epoch': 10.0}
